# Results analysis


In [ ]:
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

%matplotlib inline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS = ROOT / "reports"
FIGS = REPORTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 8.5,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "legend.frameon": False, "savefig.bbox": "tight",
})

BACKBONES = ["mobilenetv2", "mobilenetv3large", "efficientnetb0", "densenet121", "efficientnetb3",
             "resnet50", "inceptionv3", "vgg16", "vgg19", "efficientnetb5", "resnet101"]
NAMES = {"mobilenetv2": "MobileNetV2", "mobilenetv3large": "MobileNetV3Large",
         "efficientnetb0": "EfficientNetB0", "densenet121": "DenseNet121",
         "efficientnetb3": "EfficientNetB3", "resnet50": "ResNet50", "inceptionv3": "InceptionV3",
         "vgg16": "VGG16", "vgg19": "VGG19", "efficientnetb5": "EfficientNetB5", "resnet101": "ResNet101"}
CLASSES = ["abiotic_stress", "bacterial_spot", "chewing_insect", "healthy", "mechanical_stress", "mite_presence"]
SHORT = {"abiotic_stress": "Abiotic", "bacterial_spot": "Bact. spot", "chewing_insect": "Chewing",
         "healthy": "Healthy", "mechanical_stress": "Mechanical", "mite_presence": "Mite"}


def save(fig, name):
    fig.savefig(FIGS / f"{name}.png", dpi=300)
    plt.show()

In [ ]:
cbam = pd.read_csv(REPORTS / "source/cbam_vs_base.csv")
per_class = pd.read_csv(REPORTS / "source/source_per_class.csv")
conf_public = pd.read_csv(REPORTS / "source/confusion_public.csv")
typical = pd.read_csv(REPORTS / "source/typical_leaves.csv")
zero_shot = pd.read_csv(REPORTS / "target/zero_shot.csv")
cbam_orchard = pd.read_csv(REPORTS / "target/cbam_orchard.csv")
adaptation = pd.read_csv(REPORTS / "target/adaptation.csv")
baselines = pd.read_csv(REPORTS / "target/baselines.csv")
curve = pd.read_csv(REPORTS / "target/learning_curve.csv")
conf_orchard = pd.read_csv(REPORTS / "target/confusion_orchard.csv")
image_stats = pd.read_csv(REPORTS / "domain_shift/image_stats.csv")

In [ ]:
def square(path, size=256):
    img = Image.open(ROOT / "data" / path).convert("RGB")
    side = max(img.size)
    canvas = Image.new("RGB", (side, side))
    canvas.paste(img, ((side - img.width) // 2, (side - img.height) // 2))
    return canvas.resize((size, size), Image.LANCZOS)


n = 3
fig, axes = plt.subplots(len(CLASSES), 2 * n, figsize=(2 * n * 1.05, len(CLASSES) * 1.1),
                         gridspec_kw={"wspace": 0.04, "hspace": 0.1})
for i, c in enumerate(CLASSES):
    for half, domain in enumerate(["public", "orchard"]):
        paths = typical[(typical.domain == domain) & (typical["class"] == c)].sort_values("rank").relpath.tolist()
        for j in range(n):
            ax = axes[i, half * n + j]
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False)
            for s in ax.spines.values():
                s.set_visible(False)
            if j < len(paths):
                ax.imshow(square(paths[j]))
            elif j == 1:
                ax.text(0.5, 0.5, "not observed\nin the orchard", ha="center", va="center",
                        fontsize=8, color="#555555", transform=ax.transAxes)
    axes[i, 0].set_ylabel(SHORT[c], fontsize=9)
axes[0, 1].set_title("Public dataset", fontsize=10, pad=6)
axes[0, n + 1].set_title("Orchard (target domain)", fontsize=10, pad=6)
save(fig, "fig_examples")

In [ ]:
src = ROOT / "figure_sources"
cmd = ["pdflatex", "-interaction=nonstopmode", "-output-directory", str(FIGS), "fig_cbam_scheme.tex"]
subprocess.run(cmd, cwd=src, check=True, capture_output=True)
subprocess.run(["pdftoppm", "-r", "300", "-png", "-singlefile", str(FIGS / "fig_cbam_scheme.pdf"),
                str(FIGS / "fig_cbam_scheme")], check=True)
for ext in ("aux", "log", "pdf"):
    (FIGS / f"fig_cbam_scheme.{ext}").unlink(missing_ok=True)
Image.open(FIGS / "fig_cbam_scheme.png")

In [ ]:
order = [b for b in BACKBONES if b in set(cbam.backbone)]
ys = np.arange(len(order))[::-1]
panels = [(cbam, "test_ref", "test_var", "(a) Public hold-out (six classes)"),
          (cbam_orchard, "zs_ref", "zs_var", "(b) Orchard, zero-shot (four classes)")]

fig, axes = plt.subplots(1, 2, figsize=(7.6, 0.32 * len(order) + 1.0), sharey=True)
for k, (ax, (df, ref, var, title)) in enumerate(zip(axes, panels)):
    d = df.set_index("backbone").loc[order]
    ax.hlines(ys, d[ref], d[var], color=GRID, lw=2, zorder=1)
    ax.scatter(d[ref], ys, s=34, color=BLUE, edgecolor="white", linewidth=1, zorder=3,
               label="Baseline" if k == 0 else None)
    ax.scatter(d[var], ys, s=34, color=ORANGE, edgecolor="white", linewidth=1, zorder=3,
               label="+ CBAM" if k == 0 else None)
    ax.set_title(title, loc="left")
    ax.set_xlabel("Macro-F1")
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(ys, [NAMES[b] for b in order])
fig.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
fig.tight_layout()
save(fig, "fig_cbam_dumbbell")

In [ ]:
d = zero_shot[zero_shot.variant == "none"].set_index("backbone")
order = [b for b in BACKBONES if b in d.index]
d = d.loc[order]
ys = np.arange(len(order))[::-1]

fig, ax = plt.subplots(figsize=(4.6, 0.32 * len(order) + 0.9))
ax.hlines(ys, d.src4_macro_f1, d.zs_macro_f1, color=GRID, lw=2, zorder=1)
ax.scatter(d.src4_macro_f1, ys, s=34, color=BLUE, edgecolor="white", linewidth=1, zorder=3, label="Public hold-out")
ax.scatter(d.zs_macro_f1, ys, s=34, color=ORANGE, edgecolor="white", linewidth=1, zorder=3, label="Orchard (zero-shot)")
ax.set_yticks(ys, [NAMES[b] for b in order])
ax.set_xlabel("Macro-F1 on the four shared classes (baseline models)")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
save(fig, "fig_zero_shot_gap")

In [ ]:
lc = curve.copy()
full = pd.concat([adaptation[(adaptation.variant == "none") & adaptation.method.isin(["fta", "lpft"])],
                  baselines[(baselines.variant == "none") & (baselines.method == "scratch")]])
for _, r in full.iterrows():
    sel = (lc.backbone == r.backbone) & (lc.method == r.method) & (lc.n_target.astype(str) == "all")
    lc.loc[sel, ["macro_f1", "low", "high"]] = [r.macro_f1, r.macro_f1_low, r.macro_f1_high]
zs = adaptation[(adaptation.variant == "none") & (adaptation.method == "zs")].set_index("backbone").macro_f1

methods = [("fta", "Public-pretrained FTA", BLUE), ("lpft", "Public-pretrained LP-FT", GREEN),
           ("scratch", "ImageNet only (LP-FT)", ORANGE)]
bbs = list(dict.fromkeys(lc.backbone))
fig, axes = plt.subplots(1, len(bbs), figsize=(2.3 * len(bbs), 2.4), sharey=True)
for ax, b in zip(axes, bbs):
    ax.axhline(zs[b], color=INK2, lw=1.2, ls="--", label="Zero-shot")
    for m, label, color in methods:
        d = lc[(lc.backbone == b) & (lc.method == m)].sort_values("n_leaves_mean")
        ax.fill_between(d.n_leaves_mean, d.low, d.high, color=color, alpha=0.15, lw=0)
        ax.plot(d.n_leaves_mean, d.macro_f1, color=color, lw=2, marker="o", ms=4, label=label)
    ax.set_title(NAMES[b])
    ax.set_xlabel("Labelled orchard leaves")
axes[0].set_ylabel("Orchard macro-F1")
axes[0].legend(loc="lower center", bbox_to_anchor=(len(bbs) / 2 + 0.1, 1.08), ncol=4)
save(fig, "fig_learning_curve")

In [ ]:
def confusion_grid(df, name):
    panels = list(df.groupby(["model", "setting"], sort=False)) if df.setting.notna().any() \
        else list(df.groupby("model", sort=False))
    labels = list(dict.fromkeys(df.true))
    k = len(labels)
    nrows = int(np.ceil(len(panels) / 2))
    fig, axes = plt.subplots(nrows, 2, figsize=(6.2, 2.8 * nrows), squeeze=False)
    for ax, (key, d) in zip(axes.ravel(), panels):
        cm = d.pivot(index="true", columns="pred", values="mean_count").loc[labels, labels].to_numpy()
        norm = cm / np.maximum(cm.sum(1, keepdims=True), 1e-9)
        ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
        for i in range(k):
            for j in range(k):
                txt = f"{cm[i, j]:.1f}" if cm[i, j] % 1 else f"{int(cm[i, j])}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=7.5,
                        color="white" if norm[i, j] > 0.55 else INK)
        ax.set_xticks(range(k), [SHORT[c] for c in labels], rotation=40, ha="right")
        ax.set_yticks(range(k), [f"{SHORT[c]} ({int(round(cm[i].sum()))})" for i, c in enumerate(labels)])
        ax.set_title(", ".join(key) if isinstance(key, tuple) else key)
        ax.set_xlabel("Predicted")
        ax.grid(False)
    for ax in axes.ravel()[len(panels):]:
        ax.axis("off")
    axes[0, 0].set_ylabel("True (support)")
    fig.tight_layout()
    save(fig, name)

In [ ]:
confusion_grid(conf_public, "fig_source_confusion")

In [ ]:
piv = per_class.pivot_table(index="backbone", columns=["class", "variant"], values="f1")
order = [b for b in BACKBONES if b in piv.index]
classes = [c for c in CLASSES if c in set(per_class["class"])]
delta = np.array([[piv.loc[b, (c, "cbam")] - piv.loc[b, (c, "none")] for c in classes] for b in order])
support = per_class.drop_duplicates("class").set_index("class")["support"]

fig, ax = plt.subplots(figsize=(4.8, 0.3 * len(order) + 1.2))
lim = max(0.05, np.nanmax(np.abs(delta)))
im = ax.imshow(delta, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
for i in range(len(order)):
    for j in range(len(classes)):
        ax.text(j, i, f"{delta[i, j]:+.2f}", ha="center", va="center", fontsize=7, color=INK)
ax.set_xticks(range(len(classes)), [f"{SHORT[c]}\n(n={support[c]})" for c in classes])
ax.set_yticks(range(len(order)), [NAMES[b] for b in order])
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cb.set_label("$\\Delta$F1 (CBAM $-$ baseline)")
save(fig, "fig_cbam_per_class")

In [ ]:
stats = ["brightness", "saturation", "hue_circular_mean", "colorfulness",
         "sharpness", "background_fraction", "megapixels", "elongation"]
groups = [("Public", image_stats.domain == "public")]
for c in sorted(image_stats.loc[image_stats.domain == "local", "campaign"].astype(str).unique()):
    groups.append((f"Orchard {c}", (image_stats.domain == "local") & (image_stats.campaign.astype(str) == c)))
colors = ["#4C72B0", "#DD8452", "#55A868"]

with plt.style.context("default"):
    fig, axes = plt.subplots(2, 4, figsize=(13, 5.6))
    for ax, col in zip(axes.ravel(), stats):
        data = [image_stats.loc[m, col].dropna().to_numpy() for _, m in groups]
        vp = ax.violinplot(data, showextrema=False)
        for body, color in zip(vp["bodies"], colors):
            body.set_facecolor(color)
            body.set_alpha(0.55)
        ax.boxplot(data, widths=0.25, showfliers=False, medianprops={"color": "k"})
        ax.set_xticks(range(1, len(groups) + 1), [g for g, _ in groups], fontsize=8)
        ax.set_title(col.replace("_", " "), fontsize=10)
        ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGS / "fig_domain_stats.png", dpi=300)
    plt.show()

In [ ]:
confusion_grid(conf_orchard, "fig_local_confusion")